# A detailed HCP example notebook

This notebook contains the entire process for ASF Imaging. 


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from asfimaging import ASFpipeline


In [ ]:
print(ASFpipeline)
print(type(ASFpipeline))

In [ ]:
# Run after installing the project (python -m pip install -e .).
# Locate this checkout from the repository root or notebooks directory.
# If working elsewhere, set path_config explicitly to your config file.
path_config = next(
    (parent / "configs/hcp_slice.json" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "configs/hcp_slice.json").is_file()),
    None,
)
if path_config is None:
    raise FileNotFoundError("Set path_config to your ASF Imaging JSON configuration")
ASFImaging = ASFpipeline(path_config)


### Generate synthetic data

In [ ]:
# 1. Generate synthetic data
X_train, y_train = ASFImaging.synthetic_train_data()

### Training RF regressor 

In [ ]:
# 2. Build model and save
RF_regressor, fitted_scaler = ASFImaging.train(X_train, y_train)

### Features from real data

In [ ]:
raw_data, raw_affine, mask = ASFImaging.data_extractor("axial")

In [ ]:
features = ASFImaging.real_feature_extractor(raw_data, raw_affine, mask)

In [ ]:
print(features.shape)

### Predict using pre-train regressor

In [ ]:
labels = ASFImaging.predict_from_features(features)

In [ ]:
dpar, dperp, asf_calculated, asf_predicted = ASFImaging.save_maps(labels, raw_image=raw_data, raw_affine=raw_affine, mask=mask)

### Visualize MAPS

In [ ]:
plt.imshow(np.squeeze(dperp).T, origin="lower")

In [ ]:
plt.imshow(np.squeeze(asf_calculated).T, origin="lower")